# ex03 · 数据预处理（对应教材 2.2）

> 这一节是后面所有章节的「数据入口」。真实数据永远有缺失值，你只需要掌握这条流水线：
> **读取 → 观察缺失 → 填充 → 独热编码 → 转 tensor**
> 答案在 `solutions/ex03-答案.md`。

In [4]:
import os
import pandas as pd
import torch

## 题 1 🌱 自己写一个带缺失值的 CSV

仿照教材，创建 4 行 4 列的「房屋数据」，**故意留两个缺失值**（用 `NaN` 表示），保存为 `house.csv`。

字段：`NumRooms, Alley, Price`（价格连续值，Alley 分类值，另加一列你自选）

**你的预测**保存成功后，`house.csv` 里的 NaN 长什么样？(NA)（用记事本/代码打开看）

In [5]:
os.makedirs(os.path.join('.', 'data'), exist_ok=True)
data_file = os.path.join('data', 'house.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price,Area\n')       # NA 表示缺失值
    f.write('NA,Pave,127500,90\n')
    f.write('2,NA,106000,75\n')
    f.write('4,NA,178100,120\n')
    f.write('NA,NA,140000,100\n')

import pandas as pd
data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price  Area
0       NaN  Pave  127500    90
1       2.0   NaN  106000    75
2       4.0   NaN  178100   120
3       NaN   NaN  140000   100


## 题 2 🌱 定位缺失值

预测：`data.isna()` 会输出什么？`data.isna().sum()` 呢？（提示：对每列统计缺失个数）

In [6]:
print(data.isna(), '\n')       #输出每个位置是否为NA，True/False
print(data.isna().sum())       #输出各个类别下的NA数量

   NumRooms  Alley  Price   Area
0      True  False  False  False
1     False   True  False  False
2     False   True  False  False
3      True   True  False  False 

NumRooms    2
Alley       3
Price       0
Area        0
dtype: int64


## 题 3 🔧 填充缺失值：先只填连续值

- **连续值**（NumRooms、Area）：用该列的**均值**填充
- **分类值**（Alley）：**先不要填**，留给题 4 处理

先手算：NumRooms 列只有 2 个非缺失值（2 和 4），均值 = 3.0

In [7]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
print('特征 inputs:\n', inputs, '\n')
print('标签 outputs:\n', outputs, '\n')

# 连续值填充均值
numeric_cols = inputs.select_dtypes(include='number').columns
inputs[numeric_cols] = inputs[numeric_cols].fillna(inputs[numeric_cols].mean())
print('连续值填充后:\n', inputs)

特征 inputs:
    NumRooms Alley
0       NaN  Pave
1       2.0   NaN
2       4.0   NaN
3       NaN   NaN 

标签 outputs:
 0    127500
1    106000
2    178100
3    140000
Name: Price, dtype: int64 

连续值填充后:
    NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


## 题 4 🔧 独热编码（one-hot）：把「缺失」当成一种类别

对 Alley 列做 `get_dummies(inputs, dummy_na=True)`。预测：结果会比原来**多几列**？每一列是什么意思？

In [8]:
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


### **【你的观察】**

- 编码后总共有3列，原来的 Alley 列变成了 Alley_Pave & Alley_nan
- `Alley_nan` 列的作用：（观察：有 3 行是 1，对应原来 Alley 缺失的那几行）
- **关键思想**：分类值缺失时，与其猜一个众数填进去，不如把「缺失」本身当成一个合法的类别——这样模型能学到「没填 Alley 的房子」和「有 Pave 的房子」是不同的。

## 题 5 🌱 pandas → tensor

用 `.values`（或 `.to_numpy()`）把 DataFrame 转成 tensor。

**思考**：为什么不能直接 `torch.tensor(inputs)`？试试直接转换会发生什么。


ValueError: could not determine the shape of object type 'DataFrame' 类型不匹配

总结：DataFrame → .values/.to_numpy() → numpy 数组 → torch.tensor()（记得指定 float32）。中间那步 numpy 是 pandas 和 torch 之间的"翻译官"。

In [10]:
X = torch.tensor(inputs.values.astype(float))
y = torch.tensor(outputs.values)
print('X:', X.shape, X.dtype)
print('y:', y.shape, y.dtype)

ValueError: could not determine the shape of object type 'DataFrame'

## 题 6 🚀 挑战：独立完成一条新流水线

**不看书**，自己构造一个「学生成绩」数据集：

- 字段：`学号(Str), 班级(A/B/C), 数学成绩, 英语成绩, 是否挂科(是/否)`
- 5 行数据，**数学成绩和班级各缺 1 个值**
- 完成：读取 → 统计缺失 → 填充（连续值均值 / 分类值：两种策略任选其一——众数填充 或 dummy_na 保留缺失类别）→ 独热编码 → 转 tensor

全部自己写，写完后对照题 1-5 的写法检查。

In [20]:
# 你的完整实现（写在这里）
os.makedirs(os.path.join('.', 'data'), exist_ok=True)
data_file = os.path.join('data', 'grade.csv')
with open(data_file, 'w') as f:
    f.write('str,class,math_grade,english_grade,fail_or_not\n')
    f.write('001,1,99,98,False\n')
    f.write('003,NA,95,85,False\n')
    f.write('009,1,91,92,False\n')
    f.write('010,3,55,65,True\n')
    f.write('025,10,NA,32,True\n')

data = pd.read_csv(data_file)
print(data)
print(data.isna().sum())
processed_data = data.fillna(data.mean())
print(processed_data)
optimised = pd.get_dummies(data, dummy_na=True)
print(optimised)
X = torch.tensor(data.values.astype(float))
print(X)

   str  class  math_grade  english_grade  fail_or_not
0    1    1.0        99.0             98        False
1    3    NaN        95.0             85        False
2    9    1.0        91.0             92        False
3   10    3.0        55.0             65         True
4   25   10.0         NaN             32         True
str              0
class            1
math_grade       1
english_grade    0
fail_or_not      0
dtype: int64
   str  class  math_grade  english_grade  fail_or_not
0    1   1.00        99.0             98        False
1    3   3.75        95.0             85        False
2    9   1.00        91.0             92        False
3   10   3.00        55.0             65         True
4   25  10.00        85.0             32         True
   str  class  math_grade  english_grade  fail_or_not
0    1    1.0        99.0             98        False
1    3    NaN        95.0             85        False
2    9    1.0        91.0             92        False
3   10    3.0        55.0   

## 【标准答案】题 6 参考实现

对照你的实现，共 4 处修正：

1. **流水线要串起来**：`fillna` 的产物要传给下一步，最后转 tensor 用的是「处理完」的 `features`，不是原始 `data`（你最后用了 `data`，所以 NaN 还留在张量里）
2. **学号是标识符**：读入时 `dtype=str` 保留字符串，然后 `drop` 掉——标识符不参与建模（你的是 int64，前导零 `001`→`1` 丢失，还混进了 `mean()`）
3. **get_dummies 要显式指定列**：默认只碰字符串列，你的数据全是数值/bool，所以它「什么都没做」。用 `columns=['class']` 指定
4. **dtype 用 float32**：训练默认 float32，不是 float64（混合 bool/float 时还要 `.astype(float)` 统一类型）


In [21]:
# ===== 标准答案：题 6 正确流水线 =====
import os
import pandas as pd
import torch

# 1. 读入：学号是标识符，显式声明为字符串（否则 001 会被读成整数 1）
data = pd.read_csv('data/grade.csv', dtype={'str': str})
print('缺失统计:\n', data.isna().sum(), '\n')

# 2. 学号是标识符，不是特征 → 单独取出，模型不用它
student_id = data['str']
features = data.drop(columns=['str'])

# 3. 连续值填均值：只填成绩列（class 是类别，不填均值）
features['math_grade'] = features['math_grade'].fillna(features['math_grade'].mean())

# 4. 独热编码：显式指定类别列（get_dummies 默认不碰数值/bool 列）
features = pd.get_dummies(features, columns=['class'], dummy_na=True)
print('编码后:\n', features, '\n')

# 5. 转 tensor：处理好的 features + .astype(float)（混合 bool/float 需统一成 float）+ float32
X = torch.tensor(features.values.astype(float), dtype=torch.float32)
print('X shape:', X.shape, 'dtype:', X.dtype)
print('还有 NaN 吗:', torch.isnan(X).any().item())   # 期望 False


缺失统计:
 str              0
class            1
math_grade       1
english_grade    0
fail_or_not      0
dtype: int64 

编码后:
    math_grade  english_grade  fail_or_not  class_1.0  class_3.0  class_10.0  \
0        99.0             98        False       True      False       False   
1        95.0             85        False      False      False       False   
2        91.0             92        False       True      False       False   
3        55.0             65         True      False       True       False   
4        85.0             32         True      False      False        True   

   class_nan  
0      False  
1       True  
2      False  
3      False  
4      False   

X shape: torch.Size([5, 7]) dtype: torch.float32
还有 NaN 吗: False
